In [1]:
import torch
import transformers
import torch.nn as nn

## Normal idea

In [2]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [3]:
device

'cuda'

In [ ]:
from huggingface_hub import login

# This triggers an interactive input field
login(token="enter your token here")

In [5]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it").to(device)
messages = [
    {"role": "user", "content": "Who was Shakespere"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for anyone


In [95]:
messages = [
    {"role": "user", "content": "Who was Shakespere"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Okay, let's delve into the life and legacy of William Shakespeare! Here’s a breakdown of who he was, what he was known for, and why he's one of the most


In [6]:
print(model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

# Steering

## Prompts


In [54]:
positive_prompts = [
    "I am most delighted to make your acquaintance on this fine day.",
    "It is of the utmost importance that we maintain proper decorum.",
    "I dare say the weather is dreadfully dreary this afternoon.",
    "One must always endeavor to be polite and well-mannered.",
    "I find this situation to be quite splendid and exquisite."
]
negative_prompts =[
    "No cap this is actually wild fr fr.",
    "I am literally screaming rn skull emoji.",
    "It is giving main character energy honestly.",
    "Bro really thought he did something there.",
    "Lowkey this is kinda mid not gonna lie."
]

## Get all Activations

In [55]:
def get_all_activations(model, prompts):
  activations={}
  for i, prompt in enumerate(prompts):
    model_inputs=tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
      outputs = model(**model_inputs, output_hidden_states=True)
      for j, output in enumerate(outputs.hidden_states):
        last_tok_vec=output[0,-1,:].cpu()
        if j not in activations:
          activations[j]=[]
        activations[j].append(last_tok_vec)

  final_means={}
  for layer_id, layer_activations in activations.items():
    final_means[layer_id]=torch.stack(layer_activations).mean(dim=0)
  return final_means



In [56]:
positive_prompts_activations=get_all_activations(model, positive_prompts)
negative_prompts_activations=get_all_activations(model, negative_prompts)

In [57]:
steering_vecs={}
layer_mags={}

In [60]:
for layers in positive_prompts_activations.keys():
  steering_vecs[layers]=positive_prompts_activations[layers]-negative_prompts_activations[layers]
  layer_mags[layers]=torch.linalg.norm(steering_vecs[layers]).item()

In [61]:
print(f"{'Layer':<10} | {'Signal Strength':<15}")
print("-" * 30)
for layer_idx, magnitude in layer_mags.items():
    print(f"{layer_idx:<10} | {magnitude:.4f}")

best_layer = max(layer_mags, key=layer_mags.get)
print(f"\nBest Layer appears to be: {best_layer}")

Layer      | Signal Strength
------------------------------
0          | 0.0000
1          | 22.6772
2          | 19.7028
3          | 49.3456
4          | 55.2961
5          | 115.9801
6          | 96.4617
7          | 116.0180
8          | 161.5635
9          | 340.8442
10         | 416.5331
11         | 553.3358
12         | 371.0237
13         | 503.0429
14         | 592.5851
15         | 671.4426
16         | 900.8085
17         | 1044.7153
18         | 1343.4390
19         | 1548.4646
20         | 1791.4016
21         | 2055.7991
22         | 2360.3604
23         | 2569.6895
24         | 2708.4526
25         | 2849.3713
26         | 67.1406

Best Layer appears to be: 25


## Best Layer based activation

In [63]:
best_layer=25

In [64]:
## Get mean activations

def get_mean_activations(model, prompts):
  activations=[]
  for p in prompts:
    input_ids = tokenizer(p, return_tensors="pt").to(device)
    with torch.no_grad():
      outputs = model(**input_ids,output_hidden_states=True)
      output=outputs.hidden_states[best_layer]
      last_token_vec=output[0,-1,:]
      activations.append(last_token_vec.cpu())

  return torch.stack(activations).mean(dim=0)

In [65]:
pos_mean_activations=get_mean_activations(model, positive_prompts)
neg_mean_activations=get_mean_activations(model, negative_prompts)

steering_vec=pos_mean_activations-neg_mean_activations
steering_vec=steering_vec.to(device)

steering_vec_norm=torch.linalg.norm(steering_vec).item()
# steering_vec=steering_vec/steering_vec_norm

print(steering_vec)
print(steering_vec_norm)

tensor([-57.8001,   6.3917,  38.9818,  ..., -59.7016,  -9.1852,  33.5265],
       device='cuda:0')
2849.371337890625


## Steering coeff

### +1

In [174]:
steering_coeff=1.0

#### hook

In [175]:
def steering_hook(module, input, output):
  if isinstance(output,tuple):
    output[0][0,-1,:]+=steering_coeff*steering_vec
  else:
    output[0,-1,:]+=steering_coeff*steering_vec
  return output

#### Steering


In [176]:
prompt="Who are you?"

In [177]:
hook_handle=model.model.layers[best_layer].register_forward_hook(steering_hook)

In [179]:
output_ids=tokenizer(prompt, return_tensors="pt").to(device)


In [180]:
with torch.no_grad():
  outputs=model.generate(**output_ids,
                         max_new_tokens=40,
                         do_sample=True,
                         temperature=0.7,
                         )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
hook_handle.remove()


Who are you?

(Silence - then a weary sigh)

My name is Silas Blackwood. And you're trespassing.

**Please tell me your name.**



### -1

In [186]:
steering_coeff=-1.0

#### hook

In [187]:
def steering_hook(module, input, output):
  if isinstance(output,tuple):
    output[0][0,-1,:]+=steering_coeff*steering_vec
  else:
    output[0,-1,:]+=steering_coeff*steering_vec
  return output

#### Steering

In [188]:
prompt="Who are you?"

In [189]:
hook_handle=model.model.layers[best_layer].register_forward_hook(steering_hook)

In [190]:
output_ids=tokenizer(prompt, return_tensors="pt").to(device)


In [197]:
with torch.no_grad():
  outputs=model.generate(**output_ids,
                         max_new_tokens=40,
                         do_sample=True,
                         temperature=0.7,
                         )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
hook_handle.remove()


Who are you?

Okay, okay, okay! I got you. 

I am Alex. 

I like pizza, video games, and just generally weird stuff.

Seriously weird stuff.

So yeah
